# Model 2 — Skill Extraction (multi-label)
**PathCompanion AI · Phase 4 (CIS 6035)**

Given a job description, predict **which skills** it requires — a **multi-label** text classifier (a text can have many skill labels at once). Two approaches, compared:
1. **Baseline** — TF-IDF + One-vs-Rest Logistic Regression
2. **Transformer** — fine-tuned DistilBERT (multi-label: sigmoid + BCE loss)

**Target:** micro-averaged F1 >= 0.70.

**Data:** LinkedIn Job Postings dataset (`arshkon/linkedin-job-postings`). Each posting already has skill labels (`job_skills`), so we use those as ground truth (a practical alternative to LLM weak-labelling).

**How to run**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Get a **Kaggle API token**: kaggle.com -> your avatar -> **Settings -> API -> Create New Token** -> downloads `kaggle.json`.
3. Run cells top-to-bottom; upload `kaggle.json` when asked.

In [ ]:
!pip -q install transformers datasets scikit-learn pandas matplotlib kagglehub

## 1. Authenticate Kaggle
Upload the `kaggle.json` you downloaded (Kaggle -> Settings -> API -> Create New Token).

In [ ]:
from google.colab import files
print("Upload kaggle.json:")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print("Kaggle auth ready.")

## 2. Download the dataset (kagglehub does the heavy download in Colab)

In [ ]:
import kagglehub, os
path = kagglehub.dataset_download("arshkon/linkedin-job-postings")
print("downloaded to:", path)

def find(name):
    for root, _, fs in os.walk(path):
        if name in fs:
            return os.path.join(root, name)
    return None

print("postings:", find("postings.csv"))
print("job_skills:", find("job_skills.csv"))
print("skills map:", find("skills.csv"))

## 3. Build the multi-label dataset
Join postings (text) with their skills, keep the most common skill categories.

In [ ]:
import pandas as pd
from collections import Counter

post = pd.read_csv(find("postings.csv"), usecols=["job_id", "title", "description"])
js = pd.read_csv(find("job_skills.csv"))            # job_id, skill_abr
skmap = pd.read_csv(find("skills.csv"))              # skill_abr, skill_name

abr2name = dict(zip(skmap["skill_abr"], skmap["skill_name"]))
js["skill"] = js["skill_abr"].map(abr2name).fillna(js["skill_abr"])
job_skills = js.groupby("job_id")["skill"].apply(set)

post = post.dropna(subset=["description"])
post["skills"] = post["job_id"].map(job_skills)
post = post.dropna(subset=["skills"])

# subset for speed
post = post.sample(n=min(12000, len(post)), random_state=42).reset_index(drop=True)

# keep the 25 most common skill categories
cnt = Counter(s for ss in post["skills"] for s in ss)
TOP = [s for s, _ in cnt.most_common(25)]
post["labels"] = post["skills"].apply(lambda ss: [s for s in ss if s in TOP])
post = post[post["labels"].map(len) > 0].reset_index(drop=True)
print(len(post), "samples;", len(TOP), "skill labels")
print("labels:", TOP)

In [ ]:
import re
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

def clean(t):
    t = re.sub(r'http\S+', ' ', str(t))
    t = re.sub(r'\s+', ' ', t)
    return t.strip()[:2000]

post["text"] = (post["title"].fillna('') + ". " + post["description"].fillna('')).apply(clean)
mlb = MultiLabelBinarizer(classes=TOP)
Y = mlb.fit_transform(post["labels"])
Xtr, Xte, Ytr, Yte = train_test_split(post["text"], Y, test_size=0.1, random_state=42)
print("train:", len(Xtr), "test:", len(Xte))

## 4. Baseline — TF-IDF + One-vs-Rest Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

vec = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)

ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(Xtr_v, Ytr)
pred = ovr.predict(Xte_v)
base_micro = f1_score(Yte, pred, average="micro")
base_macro = f1_score(Yte, pred, average="macro")
print(f"Baseline  micro-F1={base_micro:.3f}  macro-F1={base_macro:.3f}")
print(classification_report(Yte, pred, target_names=TOP, zero_division=0))

## 5. Transformer — multi-label DistilBERT (T4 GPU, ~15-25 min)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

ckpt = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(ckpt)

def tokenize(b):
    return tok(b["text"], truncation=True, padding="max_length", max_length=256)

tr_ds = Dataset.from_dict({"text": Xtr.tolist(), "labels": Ytr.astype("float32").tolist()}).map(tokenize, batched=True)
te_ds = Dataset.from_dict({"text": Xte.tolist(), "labels": Yte.astype("float32").tolist()}).map(tokenize, batched=True)

In [ ]:
import numpy as np, torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

num_labels = len(TOP)
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt, num_labels=num_labels, problem_type="multi_label_classification")

def compute_metrics(p):
    probs = 1 / (1 + np.exp(-p.predictions))
    preds = (probs >= 0.5).astype(int)
    return {"micro_f1": f1_score(p.label_ids, preds, average="micro"),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="out2", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, logging_steps=50, report_to="none",
    fp16=torch.cuda.is_available())
trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=te_ds, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
res = trainer.evaluate()
bert_micro = res["eval_micro_f1"]; bert_macro = res["eval_macro_f1"]
print(f"DistilBERT  micro-F1={bert_micro:.3f}  macro-F1={bert_macro:.3f}")

## 6. Compare

In [ ]:
import pandas as pd
print(pd.DataFrame({
    "Model": ["TF-IDF + OvR LogReg", "DistilBERT (multi-label)"],
    "micro-F1": [round(base_micro, 3), round(bert_micro, 3)],
    "macro-F1": [round(base_macro, 3), round(bert_macro, 3)],
}).to_string(index=False))

## 7. Save the model (download for the backend)

In [ ]:
import joblib, json, shutil
joblib.dump({"vectorizer": vec, "clf": ovr, "labels": list(TOP)}, "model2_skill_baseline.joblib")
json.dump(list(TOP), open("model2_labels.json", "w"))
model.save_pretrained("model2_distilbert"); tok.save_pretrained("model2_distilbert")
shutil.make_archive("model2_distilbert", "zip", "model2_distilbert")

from google.colab import files
files.download("model2_skill_baseline.joblib")
files.download("model2_labels.json")
files.download("model2_distilbert.zip")
print("Saved + downloading. Keep these for backend integration (Phase 4/5).")